# Train Downstream Fusion Models

Run this notebook after the CBM notebook has produced full-size `concept_vectors.csv` in the Drive `CLARIFY` folder.

Inputs:
- `CLARIFY/data/lyrics/concept_vectors.csv`
- `data/lyrics/master_lyrics_features.csv`
- `data/audio/master_audio_features.csv`
- `data/audio/billboard_top_100_all_years.csv` or `data/audio/top_100_billboard_songs/Billboard_Top_100_*.csv`

Outputs:
- `CLARIFY/data/lyrics/downstream_fusion_table.csv`
- `CLARIFY/lyrics/model_outputs/downstream_hit_score_models.joblib`
- `CLARIFY/lyrics/model_outputs/downstream_recommender_index.joblib`
- `CLARIFY/lyrics/model_outputs/fusion_autoencoder_model.pt`
- `CLARIFY/lyrics/model_outputs/downstream_fusion_metadata.json`

This notebook joins audio, lyric, predicted-concept, and Billboard-rank target streams; then it trains supervised hit-score models plus recommender/similarity models.

## 1. Setup

Imports sklearn, pandas, numpy, PyTorch, and joblib for downstream tabular modeling and neural fusion representation learning.

In [ ]:
!pip install -q pandas numpy scikit-learn joblib torch

In [ ]:
from pathlib import Path
import copy
import json
import math
import random
import re

import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

## 2. Paths

Finds input CSVs from the repo or Drive, reads CBM concept vectors from the Drive `CLARIFY` folder, and saves downstream artifacts back to `CLARIFY`.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/CLARIFY')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    DRIVE_ROOT,
    Path('/content/DS3-CLARIFY'),
    Path('/content/drive/MyDrive/DS3-CLARIFY'),
]
PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / 'data' / 'audio' / 'master_audio_features.csv').exists()),
    DRIVE_ROOT,
)

LYRIC_DATA_DIR = PROJECT_ROOT / 'data' / 'lyrics'
AUDIO_DATA_DIR = PROJECT_ROOT / 'data' / 'audio'
BILLBOARD_CHART_DIR = AUDIO_DATA_DIR / 'top_100_billboard_songs'
BILLBOARD_COMBINED_PATH = AUDIO_DATA_DIR / 'billboard_top_100_all_years.csv'
DRIVE_LYRIC_DATA_DIR = DRIVE_ROOT / 'data' / 'lyrics'
DRIVE_AUDIO_DATA_DIR = DRIVE_ROOT / 'data' / 'audio'
MODEL_DIR = DRIVE_ROOT / 'lyrics' / 'model_outputs'
DRIVE_LYRIC_DATA_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_AUDIO_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

LYRIC_FEATURE_PATH = LYRIC_DATA_DIR / 'master_lyrics_features.csv'
if not LYRIC_FEATURE_PATH.exists():
    LYRIC_FEATURE_PATH = LYRIC_DATA_DIR / 'final_CBM_input_data.csv'
CONCEPT_VECTORS_PATH = DRIVE_LYRIC_DATA_DIR / 'concept_vectors.csv'
if not CONCEPT_VECTORS_PATH.exists():
    CONCEPT_VECTORS_PATH = LYRIC_DATA_DIR / 'concept_vectors.csv'
AUDIO_FEATURE_PATH = AUDIO_DATA_DIR / 'master_audio_features.csv'
BILLBOARD_TARGETS_PATH = DRIVE_AUDIO_DATA_DIR / 'billboard_hit_score_targets.csv'

TARGET_COLUMN = 'HitScore'
RANK_COLUMNS = ['Billboard_Rank', 'Rank', 'Peak_Rank', 'peak_rank']

FUSION_TABLE_PATH = DRIVE_LYRIC_DATA_DIR / 'downstream_fusion_table.csv'
HIT_SCORE_MODELS_PATH = MODEL_DIR / 'downstream_hit_score_models.joblib'
HIT_SCORE_PREDICTIONS_PATH = DRIVE_LYRIC_DATA_DIR / 'downstream_hit_score_predictions.csv'
DASHBOARD_ANALYSIS_PATH = DRIVE_LYRIC_DATA_DIR / 'dashboard_song_analysis.csv'
DASHBOARD_RECOMMENDATIONS_PATH = DRIVE_LYRIC_DATA_DIR / 'dashboard_song_recommendations.csv'
DASHBOARD_PAYLOAD_PATH = DRIVE_LYRIC_DATA_DIR / 'dashboard_payload.json'
RECOMMENDER_INDEX_PATH = MODEL_DIR / 'downstream_recommender_index.joblib'
FUSION_ENCODER_PATH = MODEL_DIR / 'fusion_autoencoder_model.pt'
METADATA_PATH = MODEL_DIR / 'downstream_fusion_metadata.json'

print('Project root:', PROJECT_ROOT)
print('Drive artifact root:', DRIVE_ROOT)
for label, path in {'lyrics': LYRIC_FEATURE_PATH, 'concept_vectors': CONCEPT_VECTORS_PATH, 'audio': AUDIO_FEATURE_PATH, 'billboard_combined': BILLBOARD_COMBINED_PATH, 'billboard_charts': BILLBOARD_CHART_DIR}.items():
    print(label, path, path.exists())
    if label not in {'billboard_charts', 'billboard_combined'} and not path.exists():
        raise FileNotFoundError(f'Missing {label}: {path}')
print('Fusion table saves to:', FUSION_TABLE_PATH)
print('Downstream model artifacts save to:', MODEL_DIR)
print('Neural fusion encoder saves to:', FUSION_ENCODER_PATH)

## 3. Columns and Helpers

Defines the lyric, concept, and Librosa audio feature columns plus the title/artist join key.

In [ ]:
LYRIC_IDENTITY_COLUMNS = ['SONG_TITLE', 'ARTIST_NAME', 'SONG_ID']
HANDCRAFTED_FEATURE_COLUMNS = [
    'Word_Count', 'Unique_Word_Count', 'Repetition_Score', 'Average_Line_Length',
    'Vocabulary_Diversity', 'Title_Repetition', 'Explicit_Word_Count', 'Sentiment_Score',
    'Positive_Score', 'Negative_Score', 'Emotional_Intensity',
]
LIBROSA_FEATURE_COLUMNS = (
    ['tempo']
    + [f'mfcc_{i}' for i in range(1, 14)]
    + [f'chroma_mean_{i}' for i in range(1, 13)]
    + [f'chroma_std_{i}' for i in range(1, 13)]
    + ['spectral_centroid']
)
OPTIONAL_AUDIO_METADATA_FEATURES = ['year']

def normalize_text(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).lower())

def make_song_artist_key(df, title_col, artist_col):
    return df[title_col].map(normalize_text) + '|' + df[artist_col].map(normalize_text)

def embedding_sort_key(column):
    match = re.search(r'(\d+)$', column)
    return int(match.group(1)) if match else -1

def get_embedding_columns(df):
    return sorted([col for col in df.columns if re.fullmatch(r'BERT_Embedding_\d+', col)], key=embedding_sort_key)

def require_columns(df, required, name):
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f'{name} missing columns: {missing}')

def ensure_song_id(df, prefix):
    df = df.copy()
    if 'SONG_ID' not in df.columns:
        df['SONG_ID'] = [f'{prefix}_{idx:05d}' for idx in range(len(df))]
    return df

def add_join_occurrence(df):
    df = df.copy()
    df['_join_occurrence'] = df.groupby('_join_key').cumcount()
    return df

def load_billboard_targets(chart_dir, combined_file=None):
    if combined_file is not None and Path(combined_file).exists():
        targets = pd.read_csv(combined_file)
        rename_map = {'Song Name': 'SONG_TITLE', 'Artist Name': 'ARTIST_NAME'}
        targets = targets.rename(columns=rename_map)
        required = {'SONG_TITLE', 'ARTIST_NAME', 'Billboard_Year', 'Billboard_Rank'}
        missing = required - set(targets.columns)
        if missing:
            raise ValueError(f'{combined_file} missing columns: {sorted(missing)}')
        targets['Billboard_Year'] = pd.to_numeric(targets['Billboard_Year'], errors='coerce').astype('Int64')
        targets['Billboard_Rank'] = pd.to_numeric(targets['Billboard_Rank'], errors='coerce')
        if TARGET_COLUMN not in targets.columns:
            targets[TARGET_COLUMN] = ((101 - targets['Billboard_Rank']) / 100.0).clip(lower=0.0, upper=1.0)
        else:
            targets[TARGET_COLUMN] = pd.to_numeric(targets[TARGET_COLUMN], errors='coerce')
        targets['_chart_key'] = make_song_artist_key(targets, 'SONG_TITLE', 'ARTIST_NAME')
        print(f'Loaded combined Billboard target file: {combined_file}')
        return targets[['SONG_TITLE', 'ARTIST_NAME', 'Billboard_Year', 'Billboard_Rank', TARGET_COLUMN, '_chart_key']]

    rows = []
    chart_files = sorted(Path(chart_dir).glob('Billboard_Top_100_*.csv'))
    if not chart_files:
        print(f'No Billboard chart CSVs found in {chart_dir}.')
        return pd.DataFrame(columns=['SONG_TITLE', 'ARTIST_NAME', 'Billboard_Year', 'Billboard_Rank', TARGET_COLUMN, '_chart_key'])

    for path in chart_files:
        match = re.search(r'(\d{4})', path.stem)
        if not match:
            continue
        year = int(match.group(1))
        chart = pd.read_csv(path)
        if not {'Song Name', 'Artist Name'}.issubset(chart.columns):
            raise ValueError(f'{path} missing Song Name / Artist Name columns')
        chart = chart[['Song Name', 'Artist Name']].copy()
        chart['Billboard_Year'] = year
        chart['Billboard_Rank'] = np.arange(1, len(chart) + 1)
        chart[TARGET_COLUMN] = ((101 - chart['Billboard_Rank']) / 100.0).clip(lower=0.0, upper=1.0)
        chart = chart.rename(columns={'Song Name': 'SONG_TITLE', 'Artist Name': 'ARTIST_NAME'})
        rows.append(chart)

    targets = pd.concat(rows, ignore_index=True)
    targets['_chart_key'] = make_song_artist_key(targets, 'SONG_TITLE', 'ARTIST_NAME')
    return targets

def attach_billboard_targets(df, targets):
    df = df.copy()
    if targets.empty or 'year' not in df.columns:
        return df
    df['_chart_key'] = make_song_artist_key(df, 'SONG_TITLE', 'ARTIST_NAME')
    df['Billboard_Year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    target_lookup = targets[['_chart_key', 'Billboard_Year', 'Billboard_Rank', TARGET_COLUMN]].copy()
    out = df.merge(target_lookup, on=['_chart_key', 'Billboard_Year'], how='left')
    out = out.drop(columns=['_chart_key'])
    return out

## 4. Load Component Outputs

Loads the lyric table, CBM concept vectors, and master Librosa audio table.

In [ ]:
lyrics_df = ensure_song_id(pd.read_csv(LYRIC_FEATURE_PATH), 'lyrics_full')
concept_vectors = ensure_song_id(pd.read_csv(CONCEPT_VECTORS_PATH), 'lyrics_full')
audio_df = ensure_song_id(pd.read_csv(AUDIO_FEATURE_PATH), 'audio')

billboard_targets = load_billboard_targets(BILLBOARD_CHART_DIR, BILLBOARD_COMBINED_PATH)
billboard_targets.to_csv(BILLBOARD_TARGETS_PATH, index=False)
audio_df = attach_billboard_targets(audio_df, billboard_targets)

embedding_columns = get_embedding_columns(lyrics_df)
LYRIC_INPUT_COLUMNS = HANDCRAFTED_FEATURE_COLUMNS + embedding_columns
CONCEPT_VECTOR_COLUMNS = [
    col for col in concept_vectors.columns
    if col.startswith('Concept_') and not col.startswith('ConceptUncertainty_')
]

require_columns(lyrics_df, LYRIC_IDENTITY_COLUMNS + LYRIC_INPUT_COLUMNS, 'lyrics feature table')
require_columns(concept_vectors, LYRIC_IDENTITY_COLUMNS + CONCEPT_VECTOR_COLUMNS, 'concept vector table')
require_columns(audio_df, ['SONG_ID', 'SONG_TITLE', 'ARTIST_NAME'] + LIBROSA_FEATURE_COLUMNS, 'master audio feature table')

lyrics_df['_join_key'] = make_song_artist_key(lyrics_df, 'SONG_TITLE', 'ARTIST_NAME')
concept_vectors['_join_key'] = make_song_artist_key(concept_vectors, 'SONG_TITLE', 'ARTIST_NAME')
audio_df['_join_key'] = make_song_artist_key(audio_df, 'SONG_TITLE', 'ARTIST_NAME')
lyrics_df = add_join_occurrence(lyrics_df)
concept_vectors = add_join_occurrence(concept_vectors)
audio_df = add_join_occurrence(audio_df)

print('Lyrics rows:', len(lyrics_df))
print('Concept-vector rows:', len(concept_vectors))
print('Audio rows:', len(audio_df))
print('Billboard target rows:', len(billboard_targets))
print('Audio rows with HitScore:', int(audio_df[TARGET_COLUMN].notna().sum()) if TARGET_COLUMN in audio_df.columns else 0)
print('Saved target table:', BILLBOARD_TARGETS_PATH)
print('Lyric feature columns:', len(LYRIC_INPUT_COLUMNS))
print('Concept vector columns:', len(CONCEPT_VECTOR_COLUMNS))

## 5. Build Fusion Table

Creates the shared modeling table with separate lyric IDs and audio IDs, then saves it for inspection.

In [ ]:
def prepare_librosa_audio_features(audio_df):
    audio = audio_df.copy().rename(columns={'SONG_ID': 'AUDIO_SONG_ID'})
    feature_columns = [col for col in OPTIONAL_AUDIO_METADATA_FEATURES + LIBROSA_FEATURE_COLUMNS if col in audio.columns]
    for col in feature_columns:
        audio[col] = pd.to_numeric(audio[col], errors='coerce')
    target_columns = [col for col in ['Billboard_Year', 'Billboard_Rank', TARGET_COLUMN] if col in audio.columns]
    out = audio[['AUDIO_SONG_ID', 'SONG_TITLE', 'ARTIST_NAME', '_join_key', '_join_occurrence'] + feature_columns + target_columns].copy()
    out = out.rename(columns={col: f'Audio_{col}' for col in feature_columns})
    audio_input_columns = [f'Audio_{col}' for col in feature_columns]
    out[audio_input_columns] = out[audio_input_columns].fillna(out[audio_input_columns].median(numeric_only=True))
    return out, audio_input_columns

audio_features, AUDIO_INPUT_COLUMNS = prepare_librosa_audio_features(audio_df)
lyrics_features = lyrics_df[LYRIC_IDENTITY_COLUMNS + ['_join_key', '_join_occurrence'] + LYRIC_INPUT_COLUMNS].rename(columns={'SONG_ID': 'LYRIC_SONG_ID'})
concept_features = concept_vectors[LYRIC_IDENTITY_COLUMNS + ['_join_key', '_join_occurrence'] + CONCEPT_VECTOR_COLUMNS].rename(columns={'SONG_ID': 'LYRIC_SONG_ID'})

fusion = lyrics_features.merge(
    concept_features[['LYRIC_SONG_ID', '_join_key', '_join_occurrence'] + CONCEPT_VECTOR_COLUMNS],
    on=['LYRIC_SONG_ID', '_join_key', '_join_occurrence'],
    how='inner',
    validate='one_to_one',
)
fusion = fusion.merge(
    audio_features,
    on=['_join_key', '_join_occurrence'],
    how='inner',
    suffixes=('', '_audio'),
    validate='one_to_one',
)
fusion = fusion.drop(columns=[col for col in ['SONG_TITLE_audio', 'ARTIST_NAME_audio'] if col in fusion.columns])

audio_match_keys = set(zip(audio_df['_join_key'], audio_df['_join_occurrence']))
lyric_keys = list(zip(lyrics_df['_join_key'], lyrics_df['_join_occurrence']))
missing_audio = lyrics_df.loc[[key not in audio_match_keys for key in lyric_keys], LYRIC_IDENTITY_COLUMNS]
if len(missing_audio):
    print(f'Lyric rows without matching Librosa audio: {len(missing_audio)}')
    display(missing_audio.head(10))

fusion.to_csv(FUSION_TABLE_PATH, index=False)
print('Fusion rows:', len(fusion))
print('Fusion rows with HitScore:', int(fusion[TARGET_COLUMN].notna().sum()) if TARGET_COLUMN in fusion.columns else 0)
print('Audio feature columns:', len(AUDIO_INPUT_COLUMNS))
print('Saved fusion table:', FUSION_TABLE_PATH)
fusion[['SONG_TITLE', 'ARTIST_NAME', 'LYRIC_SONG_ID', 'AUDIO_SONG_ID'] + [col for col in ['Billboard_Year', 'Billboard_Rank', TARGET_COLUMN] if col in fusion.columns] + AUDIO_INPUT_COLUMNS[:5]].head()

## 6. Optional Hit-Score Models

If a target exists, compares `concepts_only`, `lyrics_only`, `audio_only_librosa`, `audio_lyrics`, and `audio_lyrics_concepts`.

In [ ]:
def attach_downstream_target(fusion_df, lyrics_source, audio_source):
    df = fusion_df.copy()

    if TARGET_COLUMN in df.columns:
        df[TARGET_COLUMN] = pd.to_numeric(df[TARGET_COLUMN], errors='coerce')
        return df, TARGET_COLUMN

    for rank_col in RANK_COLUMNS:
        if rank_col in df.columns:
            rank = pd.to_numeric(df[rank_col], errors='coerce')
            df[TARGET_COLUMN] = ((101 - rank) / 100.0).clip(lower=0, upper=1)
            return df, TARGET_COLUMN

    target_sources = [
        ('lyrics', lyrics_source, 'LYRIC_SONG_ID'),
        ('audio', audio_source, 'AUDIO_SONG_ID'),
    ]
    for _, source_df, fusion_id_col in target_sources:
        if 'SONG_ID' not in source_df.columns:
            continue

        if TARGET_COLUMN in source_df.columns:
            target_lookup = source_df[['SONG_ID', TARGET_COLUMN]].dropna().drop_duplicates('SONG_ID')
            df = df.merge(target_lookup, left_on=fusion_id_col, right_on='SONG_ID', how='left', suffixes=('', '_target_source'))
            df = df.drop(columns=['SONG_ID'])
            return df, TARGET_COLUMN

        for rank_col in RANK_COLUMNS:
            if rank_col in source_df.columns:
                target_lookup = source_df[['SONG_ID', rank_col]].dropna().drop_duplicates('SONG_ID')
                rank = pd.to_numeric(target_lookup[rank_col], errors='coerce')
                target_lookup[TARGET_COLUMN] = ((101 - rank) / 100.0).clip(lower=0, upper=1)
                target_lookup = target_lookup[['SONG_ID', TARGET_COLUMN]]
                df = df.merge(target_lookup, left_on=fusion_id_col, right_on='SONG_ID', how='left', suffixes=('', '_target_source'))
                df = df.drop(columns=['SONG_ID'])
                return df, TARGET_COLUMN

    return df, None

def build_supervised_model_specs(feature_count):
    specs = {
        'ridge': (
            Pipeline([('scaler', StandardScaler()), ('regressor', Ridge())]),
            {'regressor__alpha': [0.1, 1.0, 10.0, 100.0]},
        ),
        'elastic_net': (
            Pipeline([('scaler', StandardScaler()), ('regressor', ElasticNet(max_iter=10000, random_state=SEED))]),
            {'regressor__alpha': [0.001, 0.01, 0.1, 1.0], 'regressor__l1_ratio': [0.1, 0.5, 0.9]},
        ),
    }

    if feature_count > 100:
        specs['pca_ridge'] = (
            Pipeline([
                ('scaler', StandardScaler()),
                ('pca', PCA(random_state=SEED)),
                ('regressor', Ridge()),
            ]),
            {
                'pca__n_components': [32, 64, 128],
                'regressor__alpha': [1.0, 10.0, 100.0],
            },
        )

    if feature_count <= 120:
        specs['random_forest'] = (
            RandomForestRegressor(random_state=SEED, n_jobs=-1),
            {
                'n_estimators': [300, 600],
                'max_depth': [12, None],
                'min_samples_leaf': [2, 5],
            },
        )

    return specs

model_ready, target_column = attach_downstream_target(fusion, lyrics_df, audio_df)
hit_score_models = {}
hit_score_metrics = None
hit_score_predictions = None
best_hit_score_model_key = None

if target_column is None:
    print('No true downstream target found. Skipping hit-score training.')
else:
    model_df = model_ready.dropna(subset=[target_column]).reset_index(drop=True)
    if len(model_df) < 30:
        print(f'Only {len(model_df)} target rows. Skipping until more target labels exist.')
    else:
        BERT_COLUMNS = [col for col in LYRIC_INPUT_COLUMNS if col.startswith('BERT_Embedding_')]
        HANDCRAFTED_COLUMNS = [col for col in LYRIC_INPUT_COLUMNS if col not in BERT_COLUMNS]
        feature_sets = {
            'concepts_only': CONCEPT_VECTOR_COLUMNS,
            'lyrics_only': LYRIC_INPUT_COLUMNS,
            'audio_only_librosa': AUDIO_INPUT_COLUMNS,
            'audio_lyrics': AUDIO_INPUT_COLUMNS + LYRIC_INPUT_COLUMNS,
            'audio_lyrics_concepts': AUDIO_INPUT_COLUMNS + LYRIC_INPUT_COLUMNS + CONCEPT_VECTOR_COLUMNS,
            'compact_interpretable': AUDIO_INPUT_COLUMNS + HANDCRAFTED_COLUMNS + CONCEPT_VECTOR_COLUMNS,
        }
        outer_cv = KFold(n_splits=min(5, len(model_df)), shuffle=True, random_state=SEED)
        inner_cv = KFold(n_splits=min(3, len(model_df)), shuffle=True, random_state=SEED)
        rows = []

        y = model_df[target_column].astype(float)
        baseline_pred = np.repeat(y.mean(), len(y))
        rows.append({
            'Feature_Set': 'baseline_mean',
            'Model': 'mean',
            'Target': target_column,
            'Rows': len(model_df),
            'Features': 0,
            'MAE': mean_absolute_error(y, baseline_pred),
            'RMSE': mean_squared_error(y, baseline_pred) ** 0.5,
            'R2': r2_score(y, baseline_pred),
            'Best_Params': {},
        })

        for feature_name, columns in feature_sets.items():
            X = model_df[columns].astype(float)
            y = model_df[target_column].astype(float)
            for model_name, (estimator, param_grid) in build_supervised_model_specs(len(columns)).items():
                pred = np.zeros(len(model_df))
                fold_params = []
                for train_idx, test_idx in outer_cv.split(X):
                    search = GridSearchCV(
                        estimator,
                        param_grid,
                        scoring='neg_root_mean_squared_error',
                        cv=inner_cv,
                        n_jobs=-1,
                    )
                    search.fit(X.iloc[train_idx], y.iloc[train_idx])
                    pred[test_idx] = search.best_estimator_.predict(X.iloc[test_idx])
                    fold_params.append(search.best_params_)

                final_search = GridSearchCV(
                    estimator,
                    param_grid,
                    scoring='neg_root_mean_squared_error',
                    cv=inner_cv,
                    n_jobs=-1,
                )
                final_search.fit(X, y)
                model_key = f'{feature_name}_{model_name}'
                hit_score_models[model_key] = {
                    'model': final_search.best_estimator_,
                    'feature_columns': columns,
                    'feature_set': feature_name,
                    'model_name': model_name,
                    'target_column': target_column,
                    'best_params': final_search.best_params_,
                }
                rows.append({
                    'Feature_Set': feature_name,
                    'Model': model_name,
                    'Target': target_column,
                    'Rows': len(model_df),
                    'Features': len(columns),
                    'MAE': mean_absolute_error(y, pred),
                    'RMSE': mean_squared_error(y, pred) ** 0.5,
                    'R2': r2_score(y, pred),
                    'Best_Params': final_search.best_params_,
                })

        hit_score_metrics = pd.DataFrame(rows).sort_values(['RMSE', 'MAE']).reset_index(drop=True)
        best_row = hit_score_metrics[hit_score_metrics['Model'] != 'mean'].iloc[0]
        best_hit_score_model_key = f"{best_row['Feature_Set']}_{best_row['Model']}"
        best_payload = hit_score_models[best_hit_score_model_key]
        prediction_features = model_ready[best_payload['feature_columns']].astype(float)
        predicted_hit_score = np.clip(best_payload['model'].predict(prediction_features), 0.0, 1.0)

        hit_score_predictions = model_ready[
            ['SONG_TITLE', 'ARTIST_NAME', 'LYRIC_SONG_ID', 'AUDIO_SONG_ID']
            + [col for col in ['Billboard_Year', 'Billboard_Rank', target_column] if col in model_ready.columns]
        ].copy()
        hit_score_predictions['Predicted_HitScore'] = predicted_hit_score
        hit_score_predictions['HitScore_Model'] = best_hit_score_model_key
        hit_score_predictions.to_csv(HIT_SCORE_PREDICTIONS_PATH, index=False)

        fusion = fusion.merge(
            hit_score_predictions[['LYRIC_SONG_ID', 'AUDIO_SONG_ID', 'Predicted_HitScore']],
            on=['LYRIC_SONG_ID', 'AUDIO_SONG_ID'],
            how='left',
        )
        fusion.to_csv(FUSION_TABLE_PATH, index=False)

        joblib.dump(hit_score_models, HIT_SCORE_MODELS_PATH)
        print('Saved tuned hit-score models:', HIT_SCORE_MODELS_PATH)
        print('Saved hit-score predictions:', HIT_SCORE_PREDICTIONS_PATH)
        print('Best hit-score model:', best_hit_score_model_key)
        display(hit_score_metrics.round(4))

## 7. Prepare Recommender Dataset

Collapse duplicate title/artist rows before nearest-neighbor search. This prevents repeated chart entries from showing up as separate recommendations for the same song.

In [ ]:
BERT_COLUMNS = [col for col in LYRIC_INPUT_COLUMNS if col.startswith('BERT_Embedding_')]
HANDCRAFTED_COLUMNS = [col for col in LYRIC_INPUT_COLUMNS if col not in BERT_COLUMNS]

def collapse_duplicate_songs(df):
    ordered = df.sort_values(['_join_key', '_join_occurrence']).reset_index(drop=True)
    meta_cols = ['_join_key', 'SONG_TITLE', 'ARTIST_NAME', 'LYRIC_SONG_ID', 'AUDIO_SONG_ID']
    optional_meta_cols = [col for col in ['Billboard_Year', 'Billboard_Rank', TARGET_COLUMN, 'Predicted_HitScore'] if col in ordered.columns]
    numeric_cols = AUDIO_INPUT_COLUMNS + HANDCRAFTED_COLUMNS + BERT_COLUMNS + CONCEPT_VECTOR_COLUMNS + optional_meta_cols
    meta = ordered.groupby('_join_key', as_index=False).first()[meta_cols]
    numeric = ordered.groupby('_join_key', as_index=False)[numeric_cols].mean()
    counts = ordered.groupby('_join_key').size().rename('_source_row_count').reset_index()
    out = meta.merge(numeric, on='_join_key', how='inner').merge(counts, on='_join_key', how='inner')
    return out

recommender_df = collapse_duplicate_songs(fusion)
duplicate_rows_collapsed = int(len(fusion) - len(recommender_df))
print('Fusion rows before duplicate collapse:', len(fusion))
print('Recommender rows after duplicate collapse:', len(recommender_df))
print('Duplicate rows collapsed:', duplicate_rows_collapsed)
print('BERT columns:', len(BERT_COLUMNS))
print('Handcrafted lyric columns:', len(HANDCRAFTED_COLUMNS))
print('Audio columns:', len(AUDIO_INPUT_COLUMNS))
print('Concept columns:', len(CONCEPT_VECTOR_COLUMNS))

## 8. Build Multi-View and Neural Fusion Recommenders

Build several recommender variants from the same data: separate audio/lyrics/concepts baselines, weighted fusion variants, and a denoising autoencoder that learns a compact cross-modal fusion embedding. BERT and audio are reduced with PCA before KNN so the recommender is less dominated by raw high-dimensional noise.

In [ ]:
def fit_feature_block(df, columns, block_name, n_components=None):
    if not columns:
        return None, None, {'columns': [], 'n_components': 0, 'explained_variance': None}
    values = df[columns].astype(float).to_numpy()
    values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
    steps = [('scaler', StandardScaler())]
    max_components = min(values.shape[0] - 1, values.shape[1])
    if n_components is not None and max_components > 1:
        actual_components = min(n_components, max_components)
        if actual_components < values.shape[1]:
            steps.append(('pca', PCA(n_components=actual_components, random_state=SEED)))
    pipeline = Pipeline(steps)
    transformed = pipeline.fit_transform(values)
    pca = pipeline.named_steps.get('pca')
    explained = float(pca.explained_variance_ratio_.sum()) if pca is not None else None
    metadata = {
        'columns': columns,
        'raw_dimensions': len(columns),
        'output_dimensions': int(transformed.shape[1]),
        'explained_variance': explained,
    }
    return transformed, pipeline, metadata

feature_blocks = {}
block_transformers = {}
block_metadata = {}

block_specs = {
    'audio': (AUDIO_INPUT_COLUMNS, 20),
    'handcrafted': (HANDCRAFTED_COLUMNS, None),
    'bert': (BERT_COLUMNS, 64),
    'concepts': (CONCEPT_VECTOR_COLUMNS, None),
}

for block_name, (columns, n_components) in block_specs.items():
    matrix, transformer, metadata = fit_feature_block(recommender_df, columns, block_name, n_components)
    feature_blocks[block_name] = matrix
    block_transformers[block_name] = transformer
    block_metadata[block_name] = metadata

pd.DataFrame(block_metadata).T

In [ ]:
RECOMMENDER_VARIANTS = {
    'audio_only': {'audio': 1.0},
    'lyrics_only': {'handcrafted': 0.15, 'bert': 0.85},
    'concepts_only': {'concepts': 1.0},
    'balanced_fusion': {'audio': 0.30, 'handcrafted': 0.10, 'bert': 0.35, 'concepts': 0.25},
    'lyric_heavy_fusion': {'audio': 0.20, 'handcrafted': 0.10, 'bert': 0.50, 'concepts': 0.20},
    'concept_heavy_fusion': {'audio': 0.20, 'handcrafted': 0.10, 'bert': 0.25, 'concepts': 0.45},
    'audio_heavy_fusion': {'audio': 0.50, 'handcrafted': 0.05, 'bert': 0.25, 'concepts': 0.20},
}
DEFAULT_RECOMMENDER_VARIANT = 'neural_fusion_autoencoder'

def build_variant_matrix(weights):
    blocks = []
    clean_weights = {name: weight for name, weight in weights.items() if weight > 0 and feature_blocks.get(name) is not None}
    total_weight = sum(clean_weights.values())
    if total_weight <= 0:
        raise ValueError('Variant has no usable feature blocks.')
    for block_name, weight in clean_weights.items():
        blocks.append(math.sqrt(weight / total_weight) * feature_blocks[block_name])
    return np.concatenate(blocks, axis=1)

class FusionAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, latent_dim=64, dropout=0.15):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, max(hidden_dim // 2, latent_dim * 2)),
            nn.ReLU(),
            nn.LayerNorm(max(hidden_dim // 2, latent_dim * 2)),
            nn.Dropout(dropout),
            nn.Linear(max(hidden_dim // 2, latent_dim * 2), latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, max(hidden_dim // 2, latent_dim * 2)),
            nn.ReLU(),
            nn.LayerNorm(max(hidden_dim // 2, latent_dim * 2)),
            nn.Linear(max(hidden_dim // 2, latent_dim * 2), hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        return reconstructed, z

AUTOENCODER_CONFIGS = [
    {'name': 'ae_48_balanced', 'hidden_dim': 192, 'latent_dim': 48, 'dropout': 0.10, 'lr': 1e-3, 'weight_decay': 1e-4, 'noise_std': 0.04},
    {'name': 'ae_64_regularized', 'hidden_dim': 256, 'latent_dim': 64, 'dropout': 0.18, 'lr': 8e-4, 'weight_decay': 3e-4, 'noise_std': 0.06},
    {'name': 'ae_96_wide', 'hidden_dim': 384, 'latent_dim': 96, 'dropout': 0.22, 'lr': 6e-4, 'weight_decay': 5e-4, 'noise_std': 0.05},
]
AUTOENCODER_EPOCHS = 350
AUTOENCODER_PATIENCE = 45
AUTOENCODER_BATCH_SIZE = 128

def train_fusion_autoencoder(matrix, config, seed=SEED):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    indices = np.arange(len(matrix))
    rng.shuffle(indices)
    val_size = max(int(len(indices) * 0.18), 128 if len(indices) >= 800 else 1)
    val_idx = indices[:val_size]
    train_idx = indices[val_size:]

    x_train = torch.tensor(matrix[train_idx], dtype=torch.float32)
    x_val = torch.tensor(matrix[val_idx], dtype=torch.float32).to(DEVICE)
    train_loader = DataLoader(TensorDataset(x_train), batch_size=AUTOENCODER_BATCH_SIZE, shuffle=True)

    model = FusionAutoencoder(
        input_dim=matrix.shape[1],
        hidden_dim=config['hidden_dim'],
        latent_dim=config['latent_dim'],
        dropout=config['dropout'],
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    loss_fn = nn.MSELoss()

    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    best_epoch = 0
    stale_epochs = 0

    for epoch in range(1, AUTOENCODER_EPOCHS + 1):
        model.train()
        train_losses = []
        for (xb,) in train_loader:
            xb = xb.to(DEVICE)
            noisy = xb + torch.randn_like(xb) * config['noise_std']
            optimizer.zero_grad()
            reconstructed, _ = model(noisy)
            loss = loss_fn(reconstructed, xb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_reconstructed, _ = model(x_val)
            val_loss = loss_fn(val_reconstructed, x_val).item()

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            stale_epochs = 0
        else:
            stale_epochs += 1

        if stale_epochs >= AUTOENCODER_PATIENCE:
            break

    model.load_state_dict(best_state)
    return model, {
        'name': config['name'],
        'best_epoch': int(best_epoch),
        'validation_mse': float(best_val_loss),
        'train_rows': int(len(train_idx)),
        'validation_rows': int(len(val_idx)),
        **config,
    }

def encode_with_autoencoder(model, matrix):
    model.eval()
    outputs = []
    loader = DataLoader(TensorDataset(torch.tensor(matrix, dtype=torch.float32)), batch_size=512)
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            _, z = model(xb)
            outputs.append(z.cpu().numpy())
    return np.vstack(outputs)

recommender_variants = {}
for variant_name, weights in RECOMMENDER_VARIANTS.items():
    vectors = build_variant_matrix(weights)
    model = NearestNeighbors(metric='cosine', algorithm='brute')
    model.fit(vectors)
    recommender_variants[variant_name] = {
        'model': model,
        'vectors': vectors,
        'weights': weights,
        'kind': 'weighted_feature_knn',
    }

neural_base_weights = RECOMMENDER_VARIANTS['balanced_fusion']
neural_base_vectors = build_variant_matrix(neural_base_weights)
autoencoder_input_scaler = StandardScaler()
autoencoder_input = autoencoder_input_scaler.fit_transform(neural_base_vectors)

autoencoder_results = []
autoencoder_models = {}
for config_idx, config in enumerate(AUTOENCODER_CONFIGS, start=1):
    print(f"Training fusion autoencoder {config_idx}/{len(AUTOENCODER_CONFIGS)}: {config['name']}")
    ae_model, result = train_fusion_autoencoder(autoencoder_input, config, seed=SEED + config_idx)
    autoencoder_results.append(result)
    autoencoder_models[config['name']] = ae_model
    print(f"  best_epoch={result['best_epoch']} | val_mse={result['validation_mse']:.6f}")

autoencoder_search = pd.DataFrame(autoencoder_results).sort_values('validation_mse').reset_index(drop=True)
best_autoencoder_config = autoencoder_search.iloc[0].to_dict()
best_autoencoder_name = best_autoencoder_config['name']
best_autoencoder_model = autoencoder_models[best_autoencoder_name]

neural_vectors_raw = encode_with_autoencoder(best_autoencoder_model, autoencoder_input)
neural_vector_scaler = StandardScaler()
neural_vectors = neural_vector_scaler.fit_transform(neural_vectors_raw)
neural_model = NearestNeighbors(metric='cosine', algorithm='brute')
neural_model.fit(neural_vectors)
recommender_variants['neural_fusion_autoencoder'] = {
    'model': neural_model,
    'vectors': neural_vectors,
    'weights': {'source_variant': 'balanced_fusion', 'neural_autoencoder': 1.0},
    'kind': 'neural_autoencoder_knn',
    'source_weights': neural_base_weights,
    'config': best_autoencoder_config,
}

torch.save(
    {
        'model_state_dict': best_autoencoder_model.state_dict(),
        'input_dim': int(autoencoder_input.shape[1]),
        'input_scaler_mean': autoencoder_input_scaler.mean_,
        'input_scaler_scale': autoencoder_input_scaler.scale_,
        'latent_scaler_mean': neural_vector_scaler.mean_,
        'latent_scaler_scale': neural_vector_scaler.scale_,
        'best_config': best_autoencoder_config,
        'all_configs': AUTOENCODER_CONFIGS,
        'search_results': autoencoder_results,
        'source_variant': 'balanced_fusion',
        'source_weights': neural_base_weights,
        'block_metadata': block_metadata,
    },
    FUSION_ENCODER_PATH,
)

recommender_payload = {
    'default_variant': DEFAULT_RECOMMENDER_VARIANT,
    'variants': recommender_variants,
    'song_metadata': recommender_df[['SONG_TITLE', 'ARTIST_NAME', 'LYRIC_SONG_ID', 'AUDIO_SONG_ID', '_source_row_count']].reset_index(drop=True),
    'block_transformers': block_transformers,
    'block_metadata': block_metadata,
    'columns': {
        'audio': AUDIO_INPUT_COLUMNS,
        'handcrafted': HANDCRAFTED_COLUMNS,
        'bert': BERT_COLUMNS,
        'concepts': CONCEPT_VECTOR_COLUMNS,
    },
    'neural_fusion': {
        'encoder_path': str(FUSION_ENCODER_PATH),
        'search_results': autoencoder_results,
        'best_config': best_autoencoder_config,
        'input_dimensions': int(autoencoder_input.shape[1]),
        'latent_dimensions': int(neural_vectors.shape[1]),
    },
}
joblib.dump(recommender_payload, RECOMMENDER_INDEX_PATH)
print('Saved multi-view recommender index:', RECOMMENDER_INDEX_PATH)
print('Saved neural fusion encoder:', FUSION_ENCODER_PATH)
print('Variants:', list(recommender_variants.keys()))
try:
    display(autoencoder_search.round(6))
except NameError:
    print(autoencoder_search.round(6).to_string(index=False))

## 9. Inspect Recommendation Variants

Use this cell to sanity-check recommendations by feature stream. The output is not a formal metric; it is a fast way to see whether bad recommendations are coming from audio, lyrics, concepts, hand-weighted fusion, or the learned neural fusion embedding.

In [ ]:
def recommend_by_title(title, artist=None, variant=DEFAULT_RECOMMENDER_VARIANT, k=5):
    title_mask = recommender_df['SONG_TITLE'].str.lower() == str(title).lower()
    if artist is not None:
        title_mask = title_mask & (recommender_df['ARTIST_NAME'].str.lower() == str(artist).lower())
    if not title_mask.any():
        title_mask = recommender_df['SONG_TITLE'].str.lower().str.contains(str(title).lower(), regex=False, na=False)
    if not title_mask.any():
        raise ValueError(f'No song found for title: {title}')
    row_index = int(np.where(title_mask.to_numpy())[0][0])
    payload = recommender_variants[variant]
    distances, indices = payload['model'].kneighbors(
        payload['vectors'][[row_index]],
        n_neighbors=min(k + 1, len(recommender_df)),
    )
    rows = []
    for distance, idx in zip(distances[0], indices[0]):
        if idx == row_index:
            continue
        rows.append({
            'Variant': variant,
            'Query': f"{recommender_df.loc[row_index, 'SONG_TITLE']} - {recommender_df.loc[row_index, 'ARTIST_NAME']}",
            'SONG_TITLE': recommender_df.loc[idx, 'SONG_TITLE'],
            'ARTIST_NAME': recommender_df.loc[idx, 'ARTIST_NAME'],
            'Similarity': 1 - distance,
        })
    return pd.DataFrame(rows)

sample_queries = ['Hold On', 'In the End', 'Hot in Herre', 'Smells Like Teen Spirit']
sample_outputs = []
inspection_variants = ['audio_only', 'lyrics_only', 'concepts_only', 'balanced_fusion', 'neural_fusion_autoencoder']
for query in sample_queries:
    for variant in inspection_variants:
        try:
            sample_outputs.append(recommend_by_title(query, variant=variant, k=5))
        except ValueError as exc:
            print(exc)
if sample_outputs:
    display(pd.concat(sample_outputs, ignore_index=True))

## 10. Export Dashboard Payload

Saves backend-ready outputs for the frontend: CBM concept model outputs, song recommendations, and hit score. These replace placeholder genre/mood/tempo/key/sentiment outputs.

In [ ]:
DASHBOARD_RECOMMENDATION_K = 5
DASHBOARD_TOP_CONCEPT_K = 5

def concept_display_name(column):
    return (
        column.replace('Concept_', '')
        .replace('_', ' ')
        .replace('-', ' ')
        .strip()
    )

def finite_or_none(value):
    if value is None or pd.isna(value):
        return None
    return float(value)

def hit_score_to_100(value):
    value = finite_or_none(value)
    if value is None:
        return None
    return float(np.clip(value * 100.0, 0.0, 100.0))

def top_concepts_for_row(row, k=DASHBOARD_TOP_CONCEPT_K):
    concept_scores = []
    for col in CONCEPT_VECTOR_COLUMNS:
        concept_scores.append({
            'name': concept_display_name(col),
            'score': finite_or_none(row[col]),
            'column': col,
        })
    concept_scores = sorted(
        concept_scores,
        key=lambda item: -np.inf if item['score'] is None else item['score'],
        reverse=True,
    )
    return concept_scores[:k]

def build_recommendation_rows(variant=DEFAULT_RECOMMENDER_VARIANT, k=DASHBOARD_RECOMMENDATION_K):
    payload = recommender_variants[variant]
    vectors = payload['vectors']
    distances, indices = payload['model'].kneighbors(
        vectors,
        n_neighbors=min(k + 1, len(recommender_df)),
    )
    rows = []
    for query_idx in range(len(recommender_df)):
        emitted = 0
        for distance, rec_idx in zip(distances[query_idx], indices[query_idx]):
            if rec_idx == query_idx:
                continue
            rows.append({
                'query_song_id': recommender_df.loc[query_idx, 'LYRIC_SONG_ID'],
                'query_audio_id': recommender_df.loc[query_idx, 'AUDIO_SONG_ID'],
                'query_title': recommender_df.loc[query_idx, 'SONG_TITLE'],
                'query_artist': recommender_df.loc[query_idx, 'ARTIST_NAME'],
                'recommended_song_id': recommender_df.loc[rec_idx, 'LYRIC_SONG_ID'],
                'recommended_audio_id': recommender_df.loc[rec_idx, 'AUDIO_SONG_ID'],
                'recommended_title': recommender_df.loc[rec_idx, 'SONG_TITLE'],
                'recommended_artist': recommender_df.loc[rec_idx, 'ARTIST_NAME'],
                'rank': emitted + 1,
                'similarity': float(1 - distance),
                'variant': variant,
                'reason': f'{variant} similarity across audio, lyrics, and CBM concepts',
            })
            emitted += 1
            if emitted >= k:
                break
    return pd.DataFrame(rows)

dashboard_recommendations = build_recommendation_rows()
dashboard_recommendations.to_csv(DASHBOARD_RECOMMENDATIONS_PATH, index=False)

dashboard_analysis = recommender_df[
    ['SONG_TITLE', 'ARTIST_NAME', 'LYRIC_SONG_ID', 'AUDIO_SONG_ID']
    + [col for col in ['Billboard_Year', 'Billboard_Rank', TARGET_COLUMN, 'Predicted_HitScore'] if col in recommender_df.columns]
    + CONCEPT_VECTOR_COLUMNS
].copy()
dashboard_analysis = dashboard_analysis.rename(columns={
    'SONG_TITLE': 'title',
    'ARTIST_NAME': 'artist',
    'LYRIC_SONG_ID': 'song_id',
    'AUDIO_SONG_ID': 'audio_id',
    TARGET_COLUMN: 'billboard_hit_score_0_1',
    'Predicted_HitScore': 'predicted_hit_score_0_1',
})

if 'predicted_hit_score_0_1' in dashboard_analysis.columns:
    dashboard_analysis['hit_score_100'] = dashboard_analysis['predicted_hit_score_0_1'].map(hit_score_to_100)
    dashboard_analysis['hit_score_source'] = 'model_prediction'
elif 'billboard_hit_score_0_1' in dashboard_analysis.columns:
    dashboard_analysis['hit_score_100'] = dashboard_analysis['billboard_hit_score_0_1'].map(hit_score_to_100)
    dashboard_analysis['hit_score_source'] = 'billboard_rank_target'
else:
    dashboard_analysis['hit_score_100'] = np.nan
    dashboard_analysis['hit_score_source'] = 'unavailable'

dashboard_analysis['top_concepts_json'] = [
    json.dumps(top_concepts_for_row(row), ensure_ascii=False)
    for _, row in dashboard_analysis.iterrows()
]
dashboard_analysis['top_concepts'] = [
    ', '.join(item['name'] for item in top_concepts_for_row(row))
    for _, row in dashboard_analysis.iterrows()
]
dashboard_analysis.to_csv(DASHBOARD_ANALYSIS_PATH, index=False)

recommendation_groups = {
    song_id: group.sort_values('rank').to_dict(orient='records')
    for song_id, group in dashboard_recommendations.groupby('query_song_id')
}

dashboard_records = []
for _, row in dashboard_analysis.iterrows():
    song_id = row['song_id']
    top_concepts = top_concepts_for_row(row)
    hit_score_100 = finite_or_none(row.get('hit_score_100'))
    dashboard_records.append({
        'song_id': song_id,
        'audio_id': row['audio_id'],
        'title': row['title'],
        'artist': row['artist'],
        'model_outputs': {
            'type': 'cbm_concepts',
            'concepts': top_concepts,
            'top_concepts': [item['name'] for item in top_concepts],
        },
        'hit_score': {
            'score': hit_score_100,
            'score_100': hit_score_100,
            'source': row.get('hit_score_source'),
            'predicted_0_1': finite_or_none(row.get('predicted_hit_score_0_1')),
            'billboard_target_0_1': finite_or_none(row.get('billboard_hit_score_0_1')),
            'billboard_rank': finite_or_none(row.get('Billboard_Rank')),
            'billboard_year': finite_or_none(row.get('Billboard_Year')),
            'model': best_hit_score_model_key,
        },
        'recommendations': [
            {
                'title': rec['recommended_title'],
                'artist': rec['recommended_artist'],
                'song_id': rec['recommended_song_id'],
                'audio_id': rec['recommended_audio_id'],
                'rank': int(rec['rank']),
                'similarity': float(rec['similarity']),
                'reason': rec['reason'],
            }
            for rec in recommendation_groups.get(song_id, [])
        ],
    })

dashboard_payload = {
    'schema_version': 'clarify_dashboard_v1',
    'model_outputs': 'cbm_concepts',
    'default_recommender_variant': DEFAULT_RECOMMENDER_VARIANT,
    'hit_score_model': best_hit_score_model_key,
    'concept_columns': CONCEPT_VECTOR_COLUMNS,
    'records': dashboard_records,
}
DASHBOARD_PAYLOAD_PATH.write_text(
    json.dumps(dashboard_payload, indent=2, ensure_ascii=False),
    encoding='utf-8',
)

print('Saved dashboard analysis:', DASHBOARD_ANALYSIS_PATH)
print('Saved dashboard recommendations:', DASHBOARD_RECOMMENDATIONS_PATH)
print('Saved dashboard payload:', DASHBOARD_PAYLOAD_PATH)
print('Dashboard records:', len(dashboard_records))
dashboard_analysis[['title', 'artist', 'hit_score_100', 'top_concepts']].head()

## 10. Save Metadata

Writes run metadata showing row counts, feature counts, artifact paths, recommender variants, autoencoder tuning results, and whether a target was available.

In [ ]:
metadata = {
    'rows': {
        'lyrics': int(len(lyrics_df)),
        'audio': int(len(audio_df)),
        'billboard_targets': int(len(billboard_targets)),
        'audio_rows_with_hit_score': int(audio_df[TARGET_COLUMN].notna().sum()) if TARGET_COLUMN in audio_df.columns else 0,
        'concept_vectors': int(len(concept_vectors)),
        'fusion': int(len(fusion)),
        'fusion_rows_with_hit_score': int(fusion[TARGET_COLUMN].notna().sum()) if TARGET_COLUMN in fusion.columns else 0,
        'supervised_training_rows': int(len(model_ready.dropna(subset=[target_column]))) if target_column else 0,
        'recommender_unique_songs': int(len(recommender_df)),
        'duplicate_rows_collapsed': duplicate_rows_collapsed,
    },
    'feature_counts': {
        'audio_librosa': len(AUDIO_INPUT_COLUMNS),
        'handcrafted_lyrics': len(HANDCRAFTED_COLUMNS),
        'bert': len(BERT_COLUMNS),
        'concepts': len(CONCEPT_VECTOR_COLUMNS),
        'autoencoder_input_dimensions': int(autoencoder_input.shape[1]),
        'autoencoder_latent_dimensions': int(neural_vectors.shape[1]),
    },
    'target_definition': {
        'target_column': target_column,
        'formula': '(101 - Billboard_Rank) / 100',
        'interpretation': 'relative Billboard year-end rank strength among charting songs, not hit-vs-flop classification',
    },
    'hit_score_metrics': hit_score_metrics.round(6).to_dict(orient='records') if hit_score_metrics is not None else None,
        'best_hit_score_model': best_hit_score_model_key,
    'block_metadata': block_metadata,
    'recommender_variants': {
        name: {
            'kind': payload.get('kind'),
            'weights': payload.get('weights'),
            'vector_dimensions': int(payload['vectors'].shape[1]),
        }
        for name, payload in recommender_variants.items()
    },
    'default_recommender_variant': DEFAULT_RECOMMENDER_VARIANT,
    'neural_fusion_autoencoder': {
        'path': str(FUSION_ENCODER_PATH),
        'best_config': best_autoencoder_config,
        'search_results': autoencoder_results,
        'source_variant': 'balanced_fusion',
    },
    'requires_cbm_first': True,
    'join_strategy': 'normalized_song_title_plus_artist_occurrence',
    'dedupe_strategy': 'collapse_duplicate_title_artist_rows_by_mean_numeric_features',
    'uses_spotify_api': False,
    'uses_spotify_ids': False,
    'target_column': target_column,
    'hit_score_models': list(hit_score_models.keys()) if hit_score_models else [],
    'artifacts': {
        'billboard_targets': str(BILLBOARD_TARGETS_PATH),
        'fusion_table': str(FUSION_TABLE_PATH),
        'hit_score_models': str(HIT_SCORE_MODELS_PATH) if hit_score_models else None,
            'hit_score_predictions': str(HIT_SCORE_PREDICTIONS_PATH) if hit_score_predictions is not None else None,
        'recommender_index': str(RECOMMENDER_INDEX_PATH),
        'fusion_autoencoder': str(FUSION_ENCODER_PATH),
    },
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved metadata:', METADATA_PATH)
metadata